# 05.5 — PCA on the SSVI Implied-Volatility Surface + HAR Forecasting

**Project:** SSVI Surface Dynamics — Politecnico di Milano, A.Y. 2024/25

---

## Idea

Instead of forecasting SSVI parameters one by one (NB05, NB11), we forecast the **entire
surface directly**. The approach mirrors the PCA of yield curves (level, slope, curvature):

1. **Construct** the implied-variance surface `w(k, T)` on a fixed `(k, T)` grid for each day
2. **Decompose** via PCA on the training set → a small number of PCs explains >99% of variance
3. **Forecast** each PC score with a HAR model (1d / 5d / 22d components)
4. **Reconstruct** the predicted surface from predicted PC scores
5. **Evaluate** RMSE and R²_OOS vs the naive baseline (surface today = surface tomorrow)

**Why this is better than parameter-by-parameter forecasting:**
- Avoids error propagation (5 imprecise parameter forecasts → bad surface)
- PCs are orthogonal → no multicollinearity in the forecasting step
- PC1 (level) is highly persistent → easiest to forecast; PC2-PC4 are residual shape changes
- Natural analogy to Nelson-Siegel / PCA yield curve forecasting

**SSVI power-law parametrization:**

$$w(k,T) = \frac{\theta_T}{2}\left[1 + \rho\,\phi_T k + \sqrt{(\phi_T k + \rho)^2 + 1 - \rho^2}\right]$$

$$\theta_T = e^{\alpha} T^{\beta}, \qquad \phi_T = \frac{\eta}{\theta_T^{\gamma}(1+\theta_T)^{1-\gamma}}$$

ATM check: $\text{IV}(0,T) = \sqrt{\theta_T/T} = e^{\alpha/2} T^{\beta/2 - 1/2}$ ✓


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import cm
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

OUTPUT_DIR = Path("output")
PLOT_DIR   = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Surface grid  ---------------------------------------------------------------
K_GRID  = np.linspace(-0.40, 0.30, 20)      # 20 log-moneyness points
T_GRID  = np.array([30, 60, 91, 182, 365]) / 365.0   # 5 maturities
N_K, N_T   = len(K_GRID), len(T_GRID)
N_GRID     = N_K * N_T                       # 100 surface points per date
GRID_LABELS = [f"k={k:.2f}/T={int(t*365)}d" for t in T_GRID for k in K_GRID]

print(f"Surface grid: {N_K} moneyness × {N_T} maturities = {N_GRID} points")
print(f"k range: [{K_GRID[0]:.2f}, {K_GRID[-1]:.2f}]")
print(f"T range: {[int(t*365) for t in T_GRID]} days")


## Data: SSVI Parameters → Surface Matrix

For each calibrated date, reconstruct the total-variance surface `w(k,T)` on the fixed grid. We work with `w` (total variance, not IV) — it has a more linear structure which is better suited for PCA.


In [ ]:
# ── SSVI total-variance formula (power-law parametrization) ──────────────────
def ssvi_w(k_arr, T, alpha, beta, rho, eta, gamma):
    theta_T = np.exp(alpha) * T**beta
    phi_T   = eta / (theta_T**gamma * (1 + theta_T)**(1 - gamma))
    w       = (theta_T / 2) * (1 + rho*phi_T*k_arr
                                + np.sqrt((phi_T*k_arr + rho)**2 + (1 - rho**2)))
    return np.maximum(w, 1e-10)   # numerical floor

def ssvi_iv(k_arr, T, alpha, beta, rho, eta, gamma):
    w = ssvi_w(k_arr, T, alpha, beta, rho, eta, gamma)
    return np.sqrt(w / T)

def surface_vector(alpha, beta, rho, eta, gamma):
    vec = np.zeros(N_GRID)
    for j, T_ in enumerate(T_GRID):
        vec[j*N_K:(j+1)*N_K] = ssvi_w(K_GRID, T_, alpha, beta, rho, eta, gamma)
    return vec

# ── Load SSVI calibrations ────────────────────────────────────────────────────
ssvi_raw = pd.read_csv(OUTPUT_DIR / "ssvi_all_dates_clean_results.csv")
BASE_DATE = pd.Timestamp("2010-01-04")
ssvi_raw["date"] = BASE_DATE + pd.to_timedelta(ssvi_raw["time_elapsed"], unit="D")
ssvi_raw = ssvi_raw.dropna(subset=["alpha","beta","rho","eta","gamma"])
ssvi_raw = ssvi_raw.sort_values("date").reset_index(drop=True)

print(f"Calibrated dates: {len(ssvi_raw)}")
print(f"Range: {ssvi_raw['date'].min().date()} — {ssvi_raw['date'].max().date()}")

# ── Build surface matrix ──────────────────────────────────────────────────────
print("Building surface matrix... ", end="", flush=True)
W_mat = np.stack([surface_vector(r.alpha, r.beta, r.rho, r.eta, r.gamma)
                  for r in ssvi_raw.itertuples(index=False)], axis=0)
print(f"done.  Shape: {W_mat.shape}")

# Quick sanity: ATM at T=30d (first block, k=0 = index 10 on 20-point grid)
k0_idx  = np.argmin(np.abs(K_GRID))    # index of k≈0
atm_30  = np.sqrt(W_mat[:, k0_idx] / T_GRID[0])   # sqrt(w(0,T30)/T30) = IV_ATM(30d)
atm_ssvi = np.exp(ssvi_raw["alpha"]/2) * T_GRID[0]**(ssvi_raw["beta"]/2 - 0.5)
print(f"ATM IV cross-check (mean): surface={atm_30.mean():.4f}  formula={atm_ssvi.mean():.4f}")


In [ ]:
# ── Temporal 80/20 split ─────────────────────────────────────────────────────
N      = len(ssvi_raw)
SPLIT  = int(N * 0.80)
idx_tr = np.arange(SPLIT)
idx_te = np.arange(SPLIT, N)

W_train = W_mat[idx_tr]
W_test  = W_mat[idx_te]
dates_tr = ssvi_raw["date"].iloc[idx_tr].values
dates_te = ssvi_raw["date"].iloc[idx_te].values

print(f"Train: {len(idx_tr)} dates  ({pd.Timestamp(dates_tr[0]).date()} — {pd.Timestamp(dates_tr[-1]).date()})")
print(f"Test : {len(idx_te)} dates  ({pd.Timestamp(dates_te[0]).date()} — {pd.Timestamp(dates_te[-1]).date()})")
print(f"w statistics (train): mean={W_train.mean():.4f}  std={W_train.std():.4f}")


## PCA on the Implied-Variance Surface

Fit PCA on the training set only (anti-leakage). We keep the minimum number of components explaining ≥99% of variance.


In [ ]:
# ── PCA: fit on train only ────────────────────────────────────────────────────
N_COMPONENTS = 10   # upper bound; we'll truncate by explained variance
pca = PCA(n_components=N_COMPONENTS)
pca.fit(W_train)

# ── Explained variance ────────────────────────────────────────────────────────
ev_ratio    = pca.explained_variance_ratio_
ev_cum      = np.cumsum(ev_ratio)
n_99        = int(np.searchsorted(ev_cum, 0.99)) + 1
n_999       = int(np.searchsorted(ev_cum, 0.999)) + 1
N_PC        = n_99   # components to use for forecasting

print("Explained variance by PC:")
for i in range(min(8, N_COMPONENTS)):
    bar = "#" * int(ev_ratio[i]*100)
    print(f"  PC{i+1:2d}: {ev_ratio[i]*100:5.2f}%  (cum={ev_cum[i]*100:6.3f}%)  {bar}")
print(f"
  99.0% variance explained by {n_99} PCs")
print(f"  99.9% variance explained by {n_999} PCs")
print(f"  Using N_PC = {N_PC} components for forecasting")

# ── PC scores (projections) ───────────────────────────────────────────────────
# Truncate to N_PC components
pca_trunc  = PCA(n_components=N_PC)
pca_trunc.fit(W_train)
scores_tr  = pca_trunc.transform(W_train)   # (n_train, N_PC)
scores_all = pca_trunc.transform(W_mat)      # (n_all,   N_PC)  — for lag construction
scores_te  = scores_all[idx_te]             # (n_test,  N_PC)
print(f"
PC score matrix: train={scores_tr.shape}  test={scores_te.shape}")


In [ ]:
# ── Plot: Explained variance + PC loadings as surface shapes ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: scree plot
ax = axes[0]
ax.bar(range(1, N_COMPONENTS+1), ev_ratio*100, color="#4477aa", alpha=0.8)
ax.plot(range(1, N_COMPONENTS+1), ev_cum*100, "ko-", ms=4, lw=1.2, label="Cumulative")
ax.axhline(99,  color="red",    ls="--", lw=0.8, label="99%")
ax.axhline(99.9,color="orange", ls="--", lw=0.8, label="99.9%")
ax.set_xlabel("PC"); ax.set_ylabel("Explained variance (%)"); ax.set_title("Scree plot")
ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Right: PC1–PC4 loadings reshaped as smile at T=30d
ax = axes[1]
colors_pc = ["#cc3311","#4477aa","#229955","#884499","#e08030"]
for i in range(min(N_PC, 5)):
    loading = pca_trunc.components_[i][:N_K]  # first maturity slice
    ax.plot(K_GRID, loading, color=colors_pc[i], lw=1.5, label=f"PC{i+1}")
ax.axhline(0, color="black", lw=0.6, ls="--")
ax.set_xlabel("Log-moneyness k"); ax.set_ylabel("Loading on w(k, T=30d)")
ax.set_title("PC loadings — T=30d slice")
ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("PCA of SSVI Implied-Variance Surface", fontsize=12, fontweight="bold")
plt.tight_layout()
fpath = PLOT_DIR / "pca_surface_scree_loadings.png"
plt.savefig(fpath, dpi=130, bbox_inches="tight"); plt.show()
print(f"Saved: {fpath}")


## Economic Interpretation of Principal Components

By analogy with PCA of yield curves (Nelson-Siegel):

| PC | Shape at fixed T | Economic interpretation | Analog in SSVI params |
|----|-----------------|------------------------|-----------------------|
| **PC1** | Parallel level shift of entire surface | Overall implied variance level — tracks VIX | α (level) |
| **PC2** | Slope change across maturities | Term-structure steepening/flattening | β (term structure) |
| **PC3** | Curvature / smile width change | Tail-risk / vol-of-vol | η, γ (curvature) |
| **PC4+** | Skew asymmetry | Crash-risk direction change | ρ (leverage) |

PC1 alone explains the vast majority of surface variance — consistent with the high persistence of the volatility level. PC2-PC4 capture the orthogonal **shape** changes that SSVI parameters modulate independently.

**Key implication for forecasting:** PC1 is the easiest to forecast (it follows the HAR persistence structure of RV). PC2–PC4 encode surface shape changes that are close to white-noise at daily frequency — consistent with NB11's finding that individual SSVI parameter changes are near-unpredictable.


In [ ]:
# ── Plot: PC scores over time ─────────────────────────────────────────────────
n_plot = min(N_PC, 4)
fig, axes = plt.subplots(n_plot, 1, figsize=(13, 2.5*n_plot), sharex=True)
if n_plot == 1:
    axes = [axes]

all_dates = pd.to_datetime(ssvi_raw["date"].values)
split_date = pd.Timestamp(dates_te[0])

for i, ax in enumerate(axes):
    ax.plot(all_dates, scores_all[:, i], lw=0.7, color="#4477aa", alpha=0.85)
    ax.axvline(split_date, color="red", lw=1.2, ls="--", label="Train/Test split")
    ax.set_ylabel(f"PC{i+1} score")
    ax.set_title(f"PC{i+1}  (explains {pca_trunc.explained_variance_ratio_[i]*100:.1f}%)", fontsize=9)
    ax.grid(True, alpha=0.2)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle("PC Score Time Series (2010–2020)", fontsize=11, fontweight="bold")
plt.tight_layout()
fpath = PLOT_DIR / "pca_surface_pc_scores.png"
plt.savefig(fpath, dpi=130, bbox_inches="tight"); plt.show()
print(f"Saved: {fpath}")


## HAR Forecasting on PC Scores

For each PC score $s_i(t)$ and each horizon $h \in \{1, 5, 20\}$ we fit:

$$s_i(t+h) = \beta_0 + \beta_1 s_i(t) + \beta_2 \bar{s}_i^{(5)}(t) + \beta_3 \bar{s}_i^{(22)}(t) + \varepsilon_t$$

where $\bar{s}_i^{(n)}(t) = \tfrac{1}{n}\sum_{j=0}^{n-1} s_i(t-j)$ is the rolling mean (backward-looking only).

The **naive baseline** for each horizon is $\hat{s}_i(t+h) = s_i(t)$ (current score persists).

Standard errors: HAC Newey-West with maxlags = h.


In [ ]:
HORIZONS = [1, 5, 20]

# ── Build HAR features for each PC score ─────────────────────────────────────
def build_har_df(scores_series, min_lag=22):
    s = pd.Series(scores_series)
    df = pd.DataFrame({
        "s":      s,
        "s_5d":   s.rolling(5,  min_periods=5).mean(),
        "s_22d":  s.rolling(22, min_periods=22).mean(),
    })
    return df.iloc[min_lag:]   # drop initial NaN rows

# Train on all_dates (but features from train portion only in OOS eval)
all_scores = scores_all   # shape (n_all, N_PC)
n_all = len(all_scores)

# Results storage
har_models   = {}   # {(pc, h): fitted model}
har_results  = {}   # {(pc, h): dict of metrics}
har_preds    = {}   # {(pc, h): predicted scores on test}
har_coefs    = {}   # {(pc, h): OLS coefficients HAC}

print("Fitting HAR models...")
for pc_i in range(N_PC):
    for h in HORIZONS:
        # Build features and target on full series
        har_df = build_har_df(all_scores[:, pc_i])
        # Target: score at t+h
        har_df["target"] = pd.Series(all_scores[:, pc_i]).shift(-h).values[:len(har_df)]
        har_df = har_df.dropna()

        # Align with train/test indices
        # har_df index corresponds to original rows [22:] → need to map back
        orig_idx = np.arange(22, n_all)[:len(har_df)]  # original row indices
        tr_mask  = orig_idx < SPLIT
        te_mask  = orig_idx >= SPLIT

        X_tr = har_df.loc[tr_mask, ["s","s_5d","s_22d"]].values
        y_tr = har_df.loc[tr_mask, "target"].values
        X_te = har_df.loc[te_mask, ["s","s_5d","s_22d"]].values
        y_te = har_df.loc[te_mask, "target"].values

        if len(X_tr) < 30 or len(X_te) < 5:
            continue

        # OLS fit
        reg = LinearRegression().fit(X_tr, y_tr)
        p_te = reg.predict(X_te)
        naive_te = har_df.loc[te_mask, "s"].values   # naive: current score

        mse_m  = float(mean_squared_error(y_te, p_te))
        mse_n  = float(mean_squared_error(y_te, naive_te))
        r2_oos = float(1 - mse_m / mse_n) if mse_n > 0 else float("nan")

        har_models[(pc_i, h)]  = reg
        har_preds[(pc_i, h)]   = (p_te, y_te, naive_te)
        har_results[(pc_i, h)] = {
            "MSE": mse_m, "MSE_naive": mse_n,
            "MSE_ratio": mse_m/mse_n if mse_n > 0 else float("nan"),
            "R2_OOS": r2_oos
        }

        # HAC OLS for coefficient display
        Xc = sm.add_constant(X_tr)
        ols_hac = sm.OLS(y_tr, Xc).fit(cov_type="HAC", cov_kwds={"maxlags": h})
        har_coefs[(pc_i, h)] = ols_hac

print("Done. HAR models fitted:")
print(f"  {'':10s}", end="")
for h in HORIZONS:
    print(f"  h={h:2d} R2_OOS", end="")
print()
for pc_i in range(N_PC):
    print(f"  PC{pc_i+1:2d}       ", end="")
    for h in HORIZONS:
        r2 = har_results.get((pc_i,h), {}).get("R2_OOS", float("nan"))
        print(f"  {r2:+8.4f}  ", end="")
    print()


## Surface Reconstruction and Evaluation

Predicted PC scores → inverse PCA → predicted surface `ŵ(k,T)`. Evaluate RMSE on `w` and on IV, vs naive baseline.


In [ ]:
# ── Reconstruct predicted surface from predicted PC scores ───────────────────
def reconstruct_surface(pred_scores_matrix, pca_model):
    return pca_model.inverse_transform(pred_scores_matrix)

surface_results = {}

for h in HORIZONS:
    # Build predicted score matrix for test set
    n_te = scores_te.shape[0]
    pred_scores = np.zeros((n_te, N_PC))
    naive_scores = np.zeros((n_te, N_PC))

    for pc_i in range(N_PC):
        if (pc_i, h) in har_preds:
            p_te, y_te, naive_te = har_preds[(pc_i, h)]
            # Align — har_preds may be shorter than scores_te due to h-step shift
            n_valid = min(len(p_te), n_te)
            pred_scores[:n_valid, pc_i]  = p_te[:n_valid]
            naive_scores[:n_valid, pc_i] = naive_te[:n_valid]
        else:
            pred_scores[:, pc_i]  = scores_te[:, pc_i]
            naive_scores[:, pc_i] = scores_te[:, pc_i]

    # Determine valid (non-shifted) test indices
    n_valid = n_te - h
    if n_valid < 5:
        continue

    # Actual surfaces h steps ahead
    actual_idx = idx_te[:n_valid] + h
    valid_mask = actual_idx < len(W_mat)
    actual_idx = actual_idx[valid_mask]
    W_actual = W_mat[actual_idx]             # (n_valid, N_GRID)

    # Predicted and naive surfaces
    W_pred  = reconstruct_surface(pred_scores[:len(actual_idx)],  pca_trunc)
    W_naive = reconstruct_surface(naive_scores[:len(actual_idx)], pca_trunc)

    # Clip to valid variance values
    W_pred  = np.maximum(W_pred,  1e-10)
    W_naive = np.maximum(W_naive, 1e-10)

    # RMSE on total variance w
    rmse_pred  = float(np.sqrt(np.mean((W_actual - W_pred)**2)))
    rmse_naive = float(np.sqrt(np.mean((W_actual - W_naive)**2)))

    # Convert to IV for RMSE_iv
    def to_iv(W):
        IV = np.zeros_like(W)
        for j, T_ in enumerate(T_GRID):
            IV[:, j*N_K:(j+1)*N_K] = np.sqrt(W[:, j*N_K:(j+1)*N_K] / T_)
        return IV

    IV_actual = to_iv(W_actual)
    IV_pred   = to_iv(W_pred)
    IV_naive  = to_iv(W_naive)
    rmse_iv_pred  = float(np.sqrt(np.mean((IV_actual - IV_pred)**2)))
    rmse_iv_naive = float(np.sqrt(np.mean((IV_actual - IV_naive)**2)))

    r2_w  = float(1 - (rmse_pred/rmse_naive)**2)
    r2_iv = float(1 - (rmse_iv_pred/rmse_iv_naive)**2)

    surface_results[h] = {
        "RMSE_w":       rmse_pred,
        "RMSE_w_naive": rmse_naive,
        "RMSE_iv":      rmse_iv_pred,
        "RMSE_iv_naive":rmse_iv_naive,
        "R2_OOS_w":     r2_w,
        "R2_OOS_iv":    r2_iv,
        "W_actual":  W_actual,
        "W_pred":    W_pred,
        "W_naive":   W_naive,
        "IV_actual": IV_actual,
        "IV_pred":   IV_pred,
    }

# ── Print results table ────────────────────────────────────────────────────────
print("=" * 65)
print("SURFACE FORECASTING — HAR-PCA vs Naive Baseline")
print(f"  {'h':>3}  {'RMSE_w(HAR)':>12}  {'RMSE_w(naive)':>14}  "
      f"{'R2_OOS(w)':>10}  {'R2_OOS(iv)':>10}")
print("  " + "-"*58)
for h in HORIZONS:
    if h not in surface_results:
        continue
    r = surface_results[h]
    print(f"  {h:3d}  {r['RMSE_w']:12.6f}  {r['RMSE_w_naive']:14.6f}  "
          f"{r['R2_OOS_w']:+10.4f}  {r['R2_OOS_iv']:+10.4f}")


In [ ]:
# ── Plot: HAR R²_OOS per PC and horizon ──────────────────────────────────────
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(14, 4), sharey=False)
colors_h = ["#4477aa","#229955","#cc3311"]

for ax, (h, col) in zip(axes, zip(HORIZONS, colors_h)):
    r2_vals = [har_results.get((pc_i,h),{}).get("R2_OOS", float("nan"))
               for pc_i in range(N_PC)]
    ratio_v = [har_results.get((pc_i,h),{}).get("MSE_ratio", float("nan"))
               for pc_i in range(N_PC)]
    xp = np.arange(N_PC)
    bars = ax.bar(xp, r2_vals, color=col, alpha=0.75, edgecolor="white")
    ax.axhline(0, color="black", lw=0.8, ls="--")
    for bar, r2 in zip(bars, r2_vals):
        if not np.isnan(r2):
            ax.text(bar.get_x()+bar.get_width()/2, r2+0.005,
                    f"{r2:+.3f}", ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(xp)
    ax.set_xticklabels([f"PC{i+1}" for i in range(N_PC)])
    ax.set_title(f"h = {h}  (R²_OOS per PC)", fontsize=10)
    ax.set_ylabel("R²_OOS vs naive")
    ax.grid(True, axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("HAR forecast of PC scores — R²_OOS vs naive persistence",
             fontsize=11, fontweight="bold")
plt.tight_layout()
fpath = PLOT_DIR / "pca_har_r2oos_per_pc.png"
plt.savefig(fpath, dpi=130, bbox_inches="tight"); plt.show()
print(f"Saved: {fpath}")


In [ ]:
# ── Plot: Actual vs predicted surface (3 test dates: normal, high-err, shock) ─
if 1 in surface_results:
    sr = surface_results[1]
    W_act, W_prd = sr["W_actual"], sr["W_pred"]
    errors_per_date = np.sqrt(np.mean((W_act - W_prd)**2, axis=1))
    median_err  = np.median(errors_per_date)
    normal_idx  = int(np.argmin(np.abs(errors_per_date - median_err)))
    high_err_idx = int(np.argmax(errors_per_date))
    # shock: largest PC1 change in test set
    pc1_test_changes = np.abs(np.diff(scores_te[:len(W_act), 0]))
    shock_idx = int(np.argmax(pc1_test_changes)) + 1

    sel = [(normal_idx, "Normal day"), (high_err_idx, "High-error day"), (shock_idx, "Shock day")]
    T_PLOT = T_GRID[0]   # T=30d slice for the 2D plot
    j0 = 0

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (idx, label) in zip(axes, sel):
        if idx >= len(W_act):
            ax.axis("off"); continue
        w_a = W_act[idx, j0*N_K:(j0+1)*N_K]
        w_p = W_prd[idx, j0*N_K:(j0+1)*N_K]
        w_n = sr["W_naive"][idx, j0*N_K:(j0+1)*N_K]

        iv_a = np.sqrt(w_a / T_PLOT)
        iv_p = np.sqrt(w_p / T_PLOT)
        iv_n = np.sqrt(w_n / T_PLOT)

        ax.plot(K_GRID, iv_a*100, "k-",      lw=2.0, label="Actual")
        ax.plot(K_GRID, iv_p*100, "#cc3311--", lw=1.5, ls="--", label="HAR-PCA")
        ax.plot(K_GRID, iv_n*100, "#4477aa",  lw=1.5, ls=":",  label="Naive")
        ax.set_xlabel("Log-moneyness k")
        ax.set_ylabel("IV (%) at T=30d")

        date_str = str(pd.Timestamp(dates_te[idx]).date()) if idx < len(dates_te) else f"idx={idx}"
        ax.set_title(f"{label}
{date_str}", fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.2)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    plt.suptitle("Predicted vs Actual IV Smile at T=30d  (h=1)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    fpath = PLOT_DIR / "pca_har_surface_reconstruction.png"
    plt.savefig(fpath, dpi=130, bbox_inches="tight"); plt.show()
    print(f"Saved: {fpath}")


In [ ]:
# ── Plot: Rolling RMSE over test period ──────────────────────────────────────
if 1 in surface_results:
    sr  = surface_results[1]
    W_a = sr["W_actual"]
    W_p = sr["W_pred"]
    W_n = sr["W_naive"]
    n_v = len(W_a)

    rmse_roll_har   = pd.Series(np.sqrt(np.mean((W_a-W_p)**2, axis=1))).rolling(22).mean()
    rmse_roll_naive = pd.Series(np.sqrt(np.mean((W_a-W_n)**2, axis=1))).rolling(22).mean()

    fig, ax = plt.subplots(figsize=(13,3.5))
    dates_plot = pd.to_datetime(dates_te[:n_v])
    ax.plot(dates_plot, rmse_roll_har,   "#cc3311", lw=1.2, label="HAR-PCA (22d roll.)")
    ax.plot(dates_plot, rmse_roll_naive, "#4477aa", lw=1.2, label="Naive (22d roll.)", ls="--")
    ax.set_ylabel("RMSE on w(k,T)")
    ax.set_title("Rolling surface forecast error — HAR-PCA vs Naive  (h=1)", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    fpath = PLOT_DIR / "pca_har_rolling_rmse.png"
    plt.savefig(fpath, dpi=130, bbox_inches="tight"); plt.show()
    print(f"Saved: {fpath}")


In [ ]:
# ── HAC OLS coefficients for PC1 (most interpretable) ──────────────────────
print("="*60)
print("HAC Newey-West OLS coefficients — PC1 score forecasting")
print("(Most important PC: explains bulk of surface variance)")
for h in HORIZONS:
    key = (0, h)   # PC1 = index 0
    if key not in har_coefs:
        continue
    m = har_coefs[key]
    print(f"
  h={h}  R2_IS={m.rsquared:.4f}  R2_OOS={har_results[key]['R2_OOS']:+.4f}")
    names = ["const", "s(t)", "s_5d(t)", "s_22d(t)"]
    for name, c, t, p in zip(names, m.params, m.tvalues, m.pvalues):
        sig = "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.1 else ""))
        print(f"    {name:12s}  {c:+.4f}  (t={t:+.2f}{sig})")


## Research Findings


In [ ]:
print("="*70)
print("RESEARCH FINDINGS — PCA-HAR Surface Forecasting")
print("="*70)

print("
[1] Explained variance by N_PC PCs:")
for i in range(N_PC):
    print(f"  PC{i+1}: {pca_trunc.explained_variance_ratio_[i]*100:.2f}%  "
          f"(cum={np.cumsum(pca_trunc.explained_variance_ratio_)[i]*100:.3f}%)")

print("
[2] HAR forecast of PC scores — R2_OOS:")
for h in HORIZONS:
    pc_r2 = [har_results.get((i,h),{}).get("R2_OOS", float("nan")) for i in range(N_PC)]
    best_pc = int(np.nanargmax(pc_r2)) if not all(np.isnan(pc_r2)) else -1
    print(f"  h={h:2d}: " + "  ".join([f"PC{i+1}={r2:+.4f}" for i, r2 in enumerate(pc_r2)]))
    if best_pc >= 0:
        print(f"        Best: PC{best_pc+1} (R2_OOS={pc_r2[best_pc]:+.4f})")

print("
[3] Full surface reconstruction RMSE vs Naive:")
for h in HORIZONS:
    if h not in surface_results:
        continue
    r = surface_results[h]
    beat = "BEATS naive" if r["R2_OOS_w"] > 0 else "WORSE than naive"
    print(f"  h={h:2d}: RMSE_w={r['RMSE_w']:.6f}  naive={r['RMSE_w_naive']:.6f}  "
          f"R2_OOS={r['R2_OOS_w']:+.4f}  ({beat})")

print("
[4] Comparison to NB11 (individual parameter Δ forecasting):")
print("  PCA-HAR forecasts the whole surface jointly via orthogonal PC scores.")
print("  PC1 (level) has highest predictability — consistent with RV persistence (NB08).")
print("  PC2+ (shape) are near-random — consistent with NB11 Δ-param near-white-noise.")
print("  Surface RMSE reduction vs naive: see [3] above.")
print("
[5] Economic interpretation of PCs:")
print("  PC1 ≈ parallel level shift → analog of α (long-run variance level)")
print("  PC2 ≈ term-structure slope → analog of β (persistence/mean-reversion)")
print("  PC3 ≈ smile curvature       → analog of η, γ (vol-of-vol / tail risk)")
print("  PC4 ≈ skew asymmetry         → analog of ρ (leverage / crash-risk)")
print("="*70)
